# XXX--needs update to v1.0. 

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import healpy as hp
from rubin_sim.data import get_baseline, get_data_dir
import rubin_sim.maf as maf

In [ ]:
opsdb_file = get_baseline()
runName = os.path.split(opsdb_file)[-1].replace(".db", "")
print(runName)

In [ ]:
# Demo the map
# (not necessary for use with just the metrics - see below. This is just to demonstrate the slicepoint info)

cols = [
    "fieldRA",
    "fieldDec",
    "filter",
    "observationStartMJD",
    "visitExposureTime",
    "fiveSigmaDepth",
    "rotSkyPos",
]
simdata = maf.getSimData(opsdb_file, "night <= 365*2", cols)
simdata[0:10]

In [ ]:
slicer = maf.HealpixSlicer(nside=64)
slicer.setupSlicer(simdata)
print(f"original slicepoint keys are {slicer.slicePoints.keys()}")

In [ ]:
galmap = maf.GalacticPlanePriorityMap()
sp = galmap.run(slicer.slicePoints)

In [ ]:
# We have to make a slicePoint key for each galactic plane map component ..
print(f"new slicepoint keys are {sp.keys()}")

In [ ]:
for key in [k for k in sp.keys() if "galplane_priority" in k and "combined" in k]:
    hp.mollview(sp[key], min=0, title=key)

In [ ]:
# Okay and let's check a thing -- we've got these maps bundled up.
# Does the slicepoint really access the correct point?
# Pick a point in the plane
idx = maf.radec2pix(ra=np.radians(250), dec=np.radians(-40), nside=64)
print(idx)
slicer[idx]["slicePoint"]

In [ ]:
# Okay, let's move along to the metrics themselves

footprint_summaries = [maf.SumMetric()]
footprint_plotdicts = {"percentileClip": 95}
filter_summaries = [
    maf.MeanMetric(),
    maf.MedianMetric(),
    maf.RmsMetric(),
    maf.AreaThresholdMetric(lower_threshold=0.8),
]
filter_plotdicts = {"colorMin": 0, "colorMax": 2, "xMin": 0, "xMax": 5}
timescale_summaries = [maf.SumMetric(), maf.MedianMetric(), maf.AreaThresholdMetric(lower_threshold=0.5)]
timescale_plotdicts = {"colorMin": 0, "colorMax": 1, "xMin": 0, "xMax": 1}

science_maps = [s.replace("galplane_priority_", "").split(":")[0] for s in galmap.keynames if "sum" in s]

slicer = maf.HealpixSlicer(nside=64, useCache=False)
sql = None
bundles = {}
for m in science_maps:
    footprintmetric = maf.GalPlaneFootprintMetric(science_map=m)
    bundles[f"{m} footprint"] = maf.MetricBundle(
        footprintmetric,
        slicer,
        sql,
        plotDict=footprint_plotdicts,
        runName=runName,
        summaryMetrics=footprint_summaries,
    )
    filtermetric = maf.GalPlaneTimePerFilterMetric(science_map=m)
    bundles[f"{m} filter"] = maf.MetricBundle(
        filtermetric, slicer, sql, plotDict=filter_plotdicts, runName=runName, summaryMetrics=filter_summaries
    )
    visit_timescalesmetric = maf.GalPlaneVisitIntervalsTimescaleMetric(science_map=m)
    bundles[f"{m} visit intervals"] = maf.MetricBundle(
        visit_timescalesmetric,
        slicer,
        sql,
        plotDict=timescale_plotdicts,
        runName=runName,
        summaryMetrics=timescale_summaries,
    )
    season_timescalemetric = maf.GalPlaneSeasonGapsTimescaleMetric(science_map=m)
    bundles[f"{m} season gaps"] = maf.MetricBundle(
        season_timescalemetric,
        slicer,
        sql,
        plotDict=timescale_plotdicts,
        runName=runName,
        summaryMetrics=timescale_summaries,
    )

In [ ]:
!/bin/rm galplane_test/*
outDir = "galplane_baseline_v2.0_10yrs"
resultsDb = maf.ResultsDb(outDir)
g = maf.MetricBundleGroup(bundles, opsdb_file, outDir=outDir, resultsDb=resultsDb)
g.runAll()

In [ ]:
# Note that the 'reduce' function output is available in the bundles dictionary .. we have lots of keys now!
len(list(bundles.keys()))

In [ ]:
# Plot to the output directory
g.plotAll()

In [ ]:
plots = [p for p in bundles.keys() if "combined_map" in p]
for p in plots:
    plotDict = {"figsize": (8, 6)}
    bundles[p].setPlotDict(plotDict)
    bundles[p].plot(plotFunc=maf.HealpixSkyMap())

In [ ]:
plots = [p for p in bundles.keys() if "pencilbeams_map" in p]
for p in plots:
    plotDict = {"figsize": (8, 6)}
    bundles[p].setPlotDict(plotDict)
    bundles[p].plot(plotFunc=maf.HealpixSkyMap())

In [ ]:
rc = maf.RunComparison(runDirs=[outDir])
mdict = rc.buildMetricDict()
rc.addSummaryStats(mdict)

In [ ]:
pd.set_option("display.max_rows", None)
rc.summaryStats.T

In [ ]:
# Read in the summary stats from some other directories of interest (with the same contents)
dirs = [d for d in os.listdir(".") if "galplane" in d]

rc = maf.RunComparison(runDirs=dirs)
rc.addSummaryStats()
rc.summaryStats.T[0:10]

In [ ]:
rc.summaryStats.to_csv("galactic_plane_summary_stats.csv")

In [ ]:
# Use some of the tools in the maf.RunComparison module to help compare outputs from these multiple runs
family_runs = maf.archive.get_family_runs()
# family_runs.loc['vary_gp']
these_runs = list(family_runs.loc["vary_gp"].run.values)
these_runs += ["baseline_v2.0_10yrs"]
these_runs

In [ ]:
summaries = maf.archive.get_metric_summaries(summary_source="galactic_plane_summary_stats.csv")
# Sort the summaries in the order above --
# this should make trends with 'time spent on galactic plane' easier to see
summaries = summaries.loc[these_runs]
summaries

In [ ]:
# Identify some subsets of these metrics to make better visualizations
tau_obs = np.array([2.0, 5.0, 11.0, 46.5, 73])
tau_seasons = tau_obs * 5
tau_names = [f"Tau_{tau:.1f}".replace(".", "_") for tau in tau_obs]
tau_season_names = [f"Tau_{tau:.1f}".replace(".", "_") for tau in tau_seasons]
filterlist = ("u", "g", "r", "i", "z", "y")
metricnames = {}
for tau, tau_name in zip(tau_obs, tau_names):
    metricnames[f"footprint {tau}"] = [m for m in summaries.columns if "Footprint" in m and tau_name in m]
    metricnames[f"sum visit intervals {tau}"] = [
        m for m in summaries.columns if "VisitIntervals" in m and tau_name in m and "Sum" in m
    ]
    metricnames[f"area visit intervals {tau}"] = [
        m for m in summaries.columns if "VisitIntervals" in m and tau_name in m and "Area" in m
    ]
    metricnames[f"median visit intervals {tau}"] = [
        m for m in summaries.columns if "VisitIntervals" in m and tau_name in m and "Median" in m
    ]
for tau, tau_name in zip(tau_seasons, tau_season_names):
    metricnames[f"sum season gaps {tau}"] = [
        m for m in summaries.columns if "SeasonGaps" in m and tau_name in m and "Sum" in m
    ]
    metricnames[f"area season gaps {tau}"] = [
        m for m in summaries.columns if "SeasonGaps" in m and tau_name in m and "Area" in m
    ]
    metricnames[f"median season gaps {tau}"] = [
        m for m in summaries.columns if "SeasonGaps" in m and tau_name in m and "Median" in m
    ]
for f in filterlist:
    metricnames[f"area filter {f}"] = [
        m for m in summaries.columns if "Filter" in m and f"_{f}  HealpixSlicer" in m and "Area" in m
    ]
    metricnames[f"mean filter {f}"] = [
        m for m in summaries.columns if "Filter" in m and f"_{f}  HealpixSlicer" in m and "Mean" in m
    ]
    metricnames[f"rms filter {f}"] = [
        m for m in summaries.columns if "Filter" in m and f"_{f}  HealpixSlicer" in m and "Rms" in m
    ]
for k in metricnames:
    print(k, len(metricnames[k]))

In [ ]:
# Normalize data frame by baseline -- the methods in archive and summary_plots can help with this
df = summaries / summaries.loc["baseline_v2.0_10yrs"]
df[-4:]

In [ ]:
# Note - some of these plots show fewer than the full number of maps; this is happening because sometimes
# the *baseline* run (baseline_v2.0_10yrs) is 0 for that metric .. so the entire column become NaN
cols = [k for k in metricnames if k.startswith("footprint")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(df[metricnames[k]])

In [ ]:
# Note - some of these plots show fewer than the full number of maps; this is happening because sometimes
# the *baseline* run (baseline_v2.0_10yrs) is 0 for that metric .. so the entire column become NaN
# --- see, here I plotted the non-normalized version (some maps just have much higher values)
cols = [k for k in metricnames if k.startswith("footprint")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(summaries[metricnames[k]])

In [ ]:
cols = [k for k in metricnames if k.startswith("area filter")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(df[metricnames[k]])

In [ ]:
cols = [k for k in metricnames if k.startswith("mean filter")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(df[metricnames[k]])

In [ ]:
cols = [k for k in metricnames if k.startswith("rms filter")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(df[metricnames[k]])

In [ ]:
cols = [k for k in metricnames if k.startswith("sum visit")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(df[metricnames[k]])

In [ ]:
cols = [k for k in metricnames if k.startswith("area visit")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(df[metricnames[k]])

In [ ]:
cols = [k for k in metricnames if k.startswith("sum season")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(df[metricnames[k]])

In [ ]:
cols = [k for k in metricnames if k.startswith("area season")]
x = np.arange(0, len(these_runs), 1)
for k in cols[2:]:
    maf.plot_run_metric(df[metricnames[k]])

In [ ]:
cols = [k for k in metricnames if k.startswith("area season")]
x = np.arange(0, len(these_runs), 1)
for k in cols:
    maf.plot_run_metric(summaries[metricnames[k]])

In [ ]:
summaries.shape